In [6]:
#!/usr/bin/env python
import copy
import coffea
import numpy as np
import awkward as ak
import json
import hist
import yaml

# from mt2 import mt2

from coffea import processor
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents.methods import vector
from coffea.lumi_tools import LumiMask

# silence warnings due to using NanoGEN instead of full NanoAOD
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

from topcoffea.modules.paths import topcoffea_path
from topcoffea.modules.histEFT import HistEFT
import topcoffea.modules.eft_helper as efth
import topcoffea.modules.corrections as tc_cor
import topcoffea.modules.event_selection as tc_es

from ttbarEFT.modules.paths import ttbarEFT_path
from ttbarEFT.modules.analysis_tools import make_mt2, get_lumi
import ttbarEFT.modules.object_selection as tt_os
import ttbarEFT.modules.event_selection as tt_es
import ttbarEFT.modules.corrections as tt_cor 
from ttbarEFT.modules.processor_tools import calc_eft_weights
from ttbarEFT.modules.processor_tools import get_syst_lists

from topcoffea.modules.get_param_from_jsons import GetParam

get_tc_param = GetParam(topcoffea_path("params/params.json"))
get_tt_param = GetParam(ttbarEFT_path("params/params.json"))

NanoAODSchema.warn_missing_crossrefs = False
np.seterr(divide='ignore', invalid='ignore', over='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

# Calculate muon trig uncert

In [11]:
import correctionlib

from coffea import lookup_tools
from coffea.lookup_tools import txt_converters, rochester_lookup

from topcoffea.modules.paths import topcoffea_path
from ttbarEFT.modules.paths import ttbarEFT_path

ceval_HLT = correctionlib.CorrectionSet.from_file(ttbarEFT_path(f"data/POG/MUO/2017_UL/ScaleFactors_Muon_highPt_HLT_2017_UL_schemaV2.json"))

pt_flat = [81.4, 136, 80.5, 93.3, 146, 59.1, 72.1, 78, 85.1, 64.4]
abseta_flat = [0.702, 0.534, 1.09, 0.806, 0.948, 0.698, 0.821, 1.71, 0.463, 1.32]

MCeff_nom_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_MCeff"].evaluate(abseta_flat, pt_flat, "nominal")
DATAeff_nom_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_DATAeff"].evaluate(abseta_flat, pt_flat, "nominal")

MCeff_up_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_MCeff"].evaluate(abseta_flat, pt_flat, "systup")
DATAeff_up_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_DATAeff"].evaluate(abseta_flat, pt_flat, "systup")   

MCeff_down_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_MCeff"].evaluate(abseta_flat, pt_flat, "systdown")
DATAeff_down_flat = ceval_HLT["NUM_HLT_DEN_HighPtLooseRelIsoProbes_DATAeff"].evaluate(abseta_flat, pt_flat, "systdown")

In [12]:
def get_ratio_uncertainty(num_hist, denom_hist):
    '''
    Calculates the propagated uncertainty per bin on the ratio of two historgams

    Parameters
    ----------
        num_hist (scikithep hist): numerator histogram
        denom_hist (scikithep hist): Denominator histogram
    
    Returns:
        list of uncertainties, one entry per histogram bin
    '''

    yvals_num = num_hist.values()
    yvals_denom = denom_hist.values()
    sigma_num = np.sqrt(num_hist.variances())
    sigma_denom = np.sqrt(denom_hist.variances())

    ratio = np.divide(yvals_num, yvals_denom)

    # calculation for error propagation for ratio = yavls_num/yvals_denom
    # generally, z=x/y; sigma_z = abs(z)sqrt((sigma_x/x)^2+(sigma_y/y)^2)
    sigma_y = np.multiply(np.abs(ratio), np.sqrt(np.add(np.square(np.divide(sigma_num, yvals_num)), np.square(np.divide(sigma_denom, yvals_denom)))))

    return sigma_y

In [21]:
def calculate_muontrig_uncert(m1, vartype):
    """
    type: must be DATA or MC 
    """
    if vartype not in ['DATA', 'MC']:
        raise ValueError(f'Variation type "{vartype}" should be DATA or MC.')
    
    nom = getattr(m1, f"trig_{vartype}eff_mu_nom") # eff nominal
    up = getattr(m1, f"trig_{vartype}eff_mu_up") # eff data muon1
    down = getattr(m1, f"trig_{vartype}eff_mu_down") # eff data muon1
    
    avg_variation = (np.abs(up-nom) + np.abs(down-nom))/2
    return avg_variation

def calculate_ratio_uncertainty(N, D, sigmaN, sigmaD):
    # generally, f=x/y; sigma_f = abs(f)sqrt((sigma_x/x)^2+(sigma_y/y)^2)
    return np.multiply(np.abs(N/D), np.sqrt(np.add(np.square(np.divide(sigmaN, N)), np.square(np.divide(sigmaD, D)))))

def calculate_dimuon_eff_uncert(x, y, sigmaX, sigmaY): 
    """
    calculates the uncertainty for the equation f = 1-(1-x)(1-y)
    with uncertaintites sigmaX, sigmaY which are fully correlated
    
    sigma_F = (1-y)sigmaX + (1-x)sigmaY
    """
    return np.multiply((1-y), sigmaX)+np.multiply((1-x), sigmaY)

def calculate_trigSF_mm(m1, m2, var):
    """
    SF = eff_data / eff_mc = [1-(1-ed1)*(1-ed2)]/[1-(1-em1)*(1-em2)]
    for num or denom, f=1-(1-x)(1-y) is uncert = (1-y)sigmaX + (1-x)sigmaY 
    then uncert for SF in total, f=x/y, sigma_f = abs(f)sqrt((sigma_x/x)^2+(sigma_y/y)^2)
    """
    ed1 = getattr(m1, f"trig_DATAeff_mu_nom") # eff data muon1
    ed2 = getattr(m2, f"trig_DATAeff_mu_nom") # eff data muon2
    em1 = getattr(m1, f"trig_MCeff_mu_nom")   # eff MC muon1
    em2 = getattr(m2, f"trig_MCeff_mu_nom")   # eff MC muon2
    
    # calculate the numerator and denominator of the SF, eff_data / eff_mc
    DATA_eff = 1-(1-ed1)*(1-ed2)
    MC_eff   = 1-(1-em1)*(1-em2)
    nominalSF = DATA_eff / MC_eff
        
    if var == 'nom':
        return nominalSF
        
    else: 
        # get the uncertainties on each individual efficiency 
        sigma_ed1 = calculate_muontrig_uncert(m1, "DATA")
        sigma_ed2 = calculate_muontrig_uncert(m2, "DATA")
        sigma_em1 = calculate_muontrig_uncert(m1, "MC")
        sigma_em2 = calculate_muontrig_uncert(m2, "MC")
        
        # calc the uncertainty on the numerator as a whole and denominator as a whole
        # this assumes data mc uncertainties are fully correlated to eachother, same for mc uncertainties 
        sigmaN = calculate_dimuon_eff_uncert(x=ed1, y=ed2, sigmaX=sigma_ed1, sigmaY=sigma_ed2)
        sigmaD = calculate_dimuon_eff_uncert(x=em1, y=em2, sigmaX=sigma_em1, sigmaY=sigma_em2)
    
        # calc the uncertainty of the SF = eff_data / eff_MC
        # assumes data and mc are uncorrelated to eachother
        SF_uncert = calculate_ratio_uncertainty(DATA_eff, MC_eff, sigmaN, sigmaD)
        
        if var == 'up':
            return nominalSF + SF_uncert
        elif var == 'down': 
            return nominalSF - SF_uncert
        else: 
            raise ValueError(f'Unknown variation type "{var}"! Should be nom, up or down.')
            

def calculate_trigSF_em(l0, l1, var):
        '''
        l0 : ele or mu
        l1 : ele or mu
        var: nom, up, down
        '''       
        is_mu_l0 = (abs(l0.pdgId) == 13)
        is_mu_l1 = (abs(l1.pdgId) == 13)
        
        DATA_eff = ak.where(is_mu_l0, l0.trig_DATAeff_mu_nom, l1.trig_DATAeff_mu_nom)
        MC_eff = ak.where(is_mu_l0, l0.trig_MCeff_mu_nom, l1.trig_MCeff_mu_nom)
        nominalSF = DATA_eff / MC_eff
        
        if var == 'nom': 
            return nominalSF
        
        else: 
            sigma_data = ak.where(is_mu_l0, 
                              calculate_muontrig_uncert(l0, "DATA"), 
                              calculate_muontrig_uncert(l1, "DATA"))
            sigma_mc = ak.where(is_mu_l0, 
                            calculate_muontrig_uncert(l0, "MC"), 
                            calculate_muontrig_uncert(l1, "MC"))
            SF_uncert = calculate_ratio_uncertainty(DATA_eff, MC_eff, sigma_data, sigma_mc)
        
            if var == 'up':
                return nominalSF + SF_uncert
            elif var == 'down':
                return nominalSF - SF_uncert     
            else: 
                raise ValueError(f'Unknown variation type "{var}"! Should be nom, up or down.')

In [26]:
def GetTrigSF(events, lep_cat):

    # Select events based on lepton category
    mask = events[f'is_{lep_cat}']

    leps = ak.pad_none(events.leps_pt_sorted, 2)
    l0 = leps[:,0]
    l1 = leps[:,1]
    
    l0_masked = ak.mask(l0, mask)
    l1_masked = ak.mask(l1, mask)
    
    # Initialize output arrays with 1.0 (default SF) - matches the length of the unmasked events array
    nom = ak.ones_like(events.event, dtype=float)
    up = ak.ones_like(events.event, dtype=float)
    down = ak.ones_like(events.event, dtype=float)

    if lep_cat == 'ee': 
        # Assuming simple product for ee (check if you need the OR formula here too!)
        calc_nom = l0_masked.trig_eff_ele_nom * l1_masked.trig_eff_ele_nom
        calc_up = l0_masked.trig_eff_ele_up * l1_masked.trig_eff_ele_up
        calc_down = l0_masked.trig_eff_ele_down * l1_masked.trig_eff_ele_down
        
    elif lep_cat == 'mm': 
        calc_nom = calculate_trigSF_mm(l0_masked, l1_masked, "nom")
        calc_up = calculate_trigSF_mm(l0_masked, l1_masked, "up")
        calc_down = calculate_trigSF_mm(l0_masked, l1_masked, "down")

    elif lep_cat == 'em': 
        calc_nom = calculate_trigSF_em(l0_masked, l1_masked, "nom")
        calc_up = calculate_trigSF_em(l0_masked, l1_masked, "up")
        calc_down = calculate_trigSF_em(l0_masked, l1_masked, "down")

    nom = ak.where(mask, calc_nom, 1.0)
    up = ak.where(mask, calc_up, 1.0)
    down = ak.where(mask, calc_down, 1.0)

    return nom, up, down

In [15]:
fname = "/cms/cephfs/data/store/mc/RunIISummer20UL17NanoAODv9/TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mc2017_realistic_v9-v1/2510000/F689F2EB-6A85-CE46-9AB0-BF87D586AFFD.root" 
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
    entry_stop=5000,
).events()

In [16]:
year = "2017"
isData = False
dataset = "UL17_TTTo2L2Nu"
lep_cat = "em"
run_era = None

In [17]:
######### Initialize Objects #########
met  = events.MET
ele  = events.Electron
mu   = events.Muon
tau  = events.Tau
jets = events.Jet 

In [23]:
leptonSelection = tt_os.Run2LeptonSelection()
######### Electron Selection ##########
ele['isGoodElec']=leptonSelection.is_sel_ele(ele)
ele_good = ele[ele.isGoodElec]

######### Muon Selection ##########
mu['pt'] = tt_cor.ApplyMuonPtCorr(mu, year, isData)
mu['isGoodMuon']=leptonSelection.is_sel_muon(mu)
mu_good = mu[mu.isGoodMuon]

ele_good = tt_cor.AttachElectronSF(ele_good, year)    
mu_good = tt_cor.AttachMuonSF(mu_good, year)    
ele_good = tt_cor.AttachElecTrigEff(ele_good, year)
mu_good = tt_cor.AttachMuonTrigEff(mu_good, year) 
            
leps = ak.concatenate([ele_good, mu_good], axis=1)
leps_sorted = leps[ak.argsort(leps.pt, axis=-1,ascending=False)] 

events['leps_pt_sorted'] = leps_sorted
tt_es.addLepCatMasks(events) 
tt_es.add2losMask(events, year, isData)



 inside the AttachMuonTrigEff func: 
trig_MCeff_mu_nom: [[], [], [], [], [0.952], [], [0.952], ..., [], [], [0.962, 0.952], [], [], []]
trig_DATAeff_mu_nom: [[], [], [], [], [0.925], [], [0.925], ..., [], [], [0.91, 0.925], [], [], []]
trig_MCeff_mu_up: [[], [], [], [], [0.953], [], [0.953], ..., [], [], [0.963, 0.953], [], [], []]
trig_DATAeff_mu_up: [[], [], [], [], [0.925], [], [0.925], ..., [], [], [0.911, 0.925], [], [], []]
trig_MCeff_mu_down: [[], [], [], [], [0.951], [], [0.951], ..., [], [], [0.96, 0.951], [], [], []]
trig_DATAeff_mu_down: [[], [], [], [], [0.924], [], [0.924], ..., [], [], [0.909, 0.924], [], [], []]


In [24]:
######## Create objects for dense axes ##########
leps_sorted = ak.pad_none(leps_sorted, 2)
l0 = leps_sorted[:,0]
l1 = leps_sorted[:,1]

ptll = (l0+l1).pt
mll = (l0+l1).mass

In [54]:
lep_cat = 'mm'

In [55]:
mask = events[f"is_{lep_cat}"]
nom, up, down = GetTrigSF(events, lep_cat)

In [56]:
print(f"nom: {nom[mask]}")
print(f"up: {up[mask]}")
print(f"down: {down[mask]}")

print(f"sum nom: {ak.sum(nom[mask])}")
print(f"sum up: {ak.sum(up[mask])}")
print(f"sum down: {ak.sum(down[mask])}")

print(f"up/nom: {ak.sum(up[mask])/ak.sum(nom[mask])}")
print(f"down/nom: {ak.sum(down[mask])/ak.sum(nom[mask])}")

nom: [0.992, 0.991, 0.997, 0.995, 0.997, ..., 0.995, 0.995, 0.994, 0.996, 0.995]
up: [0.996, 0.992, 0.999, 0.995, 0.997, ..., 0.996, 0.996, 0.996, 0.998, 0.995]
down: [0.988, 0.991, 0.995, 0.995, 0.996, ..., 0.995, 0.995, 0.992, 0.994, 0.995]
sum nom: 83.66074559494044
sum up: 83.76649901599389
sum down: 83.55499217388694
up/nom: 1.001264074570474
down/nom: 0.9987359254295256
